# 01. Preprocesamiento de Imágenes y Extracción de Características
## Clasificador de Animales: Gatos vs. Perros

Este notebook documenta y ejecuta el pipeline de preprocesamiento de datos e ingeniería de características para el proyecto de clasificación de imágenes.

### Objetivos:
1. **Evitar el Resize Distorsivo**: Insertar imágenes reales de resoluciones heterogéneas dentro de una matriz negra fija de $256 \times 256 \times 3$ manteniendo estrictamente la relación de aspecto original (Zero-Padding Letterbox).
2. **Extracción Vectorial de Características Visuales**:
   - **Forma y Contornos**: Descriptores Dalal-Triggs HOG (Histogram of Oriented Gradients).
   - **Color y Distribución**: Espacios RGB y HSV, medias, desviaciones estándar e histogramas de color sobre la máscara de primer plano.
   - **Textura y Nitidez**: Gradientes Sobel y varianza Laplaciana del pelaje.
   - **Estructura Espacial**: Descomposición en pirámide de cuadrantes $2 \times 2$ (Top-Left, Top-Right, Bottom-Left, Bottom-Right).
3. **Generación de la Matriz de Características**: Consolidar las 349 características tabulares por imagen para su posterior entrenamiento en los modelos de Machine Learning y Clasificador de Subespacios SVD.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Detección de directorios utilizando rutas normalizadas
candidates = [
    "D:/Materiales de Auxiliar/2. Machine Learning",
    os.path.abspath("..").replace("\\", "/"),
    os.path.abspath(".").replace("\\", "/")
]
REPO_DIR = None
for c in candidates:
    if os.path.exists(os.path.join(c, "data/processed/gatos")):
        REPO_DIR = c
        break
if REPO_DIR is None:
    REPO_DIR = "D:/Materiales de Auxiliar/2. Machine Learning"

DATA_DIR = os.path.join(REPO_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
GATOS_DIR = os.path.join(PROCESSED_DIR, "gatos")
PERROS_DIR = os.path.join(PROCESSED_DIR, "perros")

print(f"Raíz del repositorio: {REPO_DIR}")
print(f"Imágenes de gatos procesadas: {len(os.listdir(GATOS_DIR))}")
print(f"Imágenes de perros procesadas: {len(os.listdir(PERROS_DIR))}")

## 1. Función de Inserción en Matriz Negra (Letterbox Zero-Padding)

Para evitar la deformación anatómica de los animales (por ejemplo, aplastar o estirar el hocico o las orejas por un `cv2.resize` arbitrario), se calcula un factor de escala isotrópico $\alpha$:

$$\alpha = \min\left(\frac{W_{target}}{W_{orig}}, \frac{H_{target}}{H_{orig}}\right)$$

Las nuevas dimensiones son:
$$W' = \text{round}(W_{orig} \cdot \alpha), \quad H' = \text{round}(H_{orig} \cdot \alpha)$$

La imagen redimensionada se centra horizontal y verticalmente dentro de una matriz de ceros $\mathbf{C} \in \mathbb{R}^{256 \times 256 \times 3}$:
$$x_{offset} = \left\lfloor \frac{W_{target} - W'}{2} \right\rfloor, \quad y_{offset} = \left\lfloor \frac{H_{target} - H'}{2} \right\rfloor$$

In [ ]:
def insert_in_black_canvas(image_bgr, target_size=(256, 256)):
    """
    Inserta una imagen real de tamaño arbitrario en un canvas negro de target_size,
    manteniendo la relación de aspecto original (Letterbox zero padding).
    """
    h, w = image_bgr.shape[:2]
    target_h, target_w = target_size
    
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=interp)
    
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    
    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    return canvas, (x_offset, y_offset, new_w, new_h)

## 2. Demostración Visual del Acondicionamiento de Imágenes

Cargamos muestras del dataset para verificar que la relación de aspecto se preserva fielmente dentro del canvas negro de $256 \times 256$.

In [ ]:
# Ejemplo visual con una imagen de gato y una de perro
sample_cat_path = os.path.join(GATOS_DIR, "gato_000.jpg")
sample_dog_path = os.path.join(PERROS_DIR, "perro_000.jpg")

cat_img = cv2.imread(sample_cat_path)
dog_img = cv2.imread(sample_dog_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(cat_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Gato en Canvas Negro {cat_img.shape}", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(dog_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Perro en Canvas Negro {dog_img.shape}", fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Extracción de Descriptores de Forma: Dalal-Triggs HOG

El descriptor **HOG (Histogram of Oriented Gradients)** captura la estructura de bordes y formas silueteadas (orejas triangulares puntiagudas en gatos vs. hocicos redondeados u orejas caídas en perros):
1. **Gradientes espaciales**: Operadores Sobel $3 \times 3$ en $x$ e $y$:
   $$G_x = \frac{\partial I}{\partial x}, \quad G_y = \frac{\partial I}{\partial y}, \quad |\nabla I| = \sqrt{G_x^2 + G_y^2}, \quad \theta = \arctan\left(\frac{G_y}{G_x}\right) \pmod{180^\circ}$$
2. **Celdas espaciales**: División en celdas de $32 \times 32$ píxeles y cuantización angular en 8 bins.
3. **Normalización por bloques**: Bloques superpuestos de $2 \times 2$ celdas con normalización $L_2$:
   $$\mathbf{v}_{norm} = \frac{\mathbf{v}}{\sqrt{\|\mathbf{v}\|_2^2 + \epsilon}}$$

In [ ]:
def compute_hog_features(gray_img, cell_size=32, n_bins=8):
    """
    Descriptor HOG vectorizado puro en NumPy/OpenCV.
    Captura orientaciones de bordes características de orejas, hocico y ojos.
    """
    h, w = gray_img.shape
    gx = cv2.Sobel(gray_img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_img, cv2.CV_32F, 0, 1, ksize=3)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    ang = ang % 180.0
    
    bin_width = 180.0 / n_bins
    bin_idx = np.floor(ang / bin_width).astype(np.int32) % n_bins
    
    n_cells_y = h // cell_size
    n_cells_x = w // cell_size
    hist_grid = np.zeros((n_cells_y, n_cells_x, n_bins), dtype=np.float32)
    
    for y in range(n_cells_y):
        for x in range(n_cells_x):
            c_mag = mag[y*cell_size:(y+1)*cell_size, x*cell_size:(x+1)*cell_size]
            c_bin = bin_idx[y*cell_size:(y+1)*cell_size, x*cell_size:(x+1)*cell_size]
            for b in range(n_bins):
                hist_grid[y, x, b] = np.sum(c_mag[c_bin == b])
                
    # Normalización en bloques 2x2
    blocks = []
    for y in range(n_cells_y - 1):
        for x in range(n_cells_x - 1):
            block = hist_grid[y:y+2, x:x+2, :].flatten()
            norm = np.linalg.norm(block) + 1e-6
            blocks.append(block / norm)
            
    if blocks:
        return np.concatenate(blocks)
    return hist_grid.flatten()

## 4. Pipeline Completo de Características Visuales

Integramos descriptores de:
1. **Color Global**: Estadísticos RGB y HSV sobre los píxeles de primer plano (excluyendo el fondo negro).
2. **Histogramas de Color**: 12 bins para Hue ($H$), 8 bins para Saturation ($S$), 8 bins para Value ($V$).
3. **Pirámide Espacial de Cuadrantes ($2 \times 2$)**: Medias y desviaciones RGB en Top-Left, Top-Right, Bottom-Left y Bottom-Right.
4. **Textura y Nitidez**: Magnitud media de gradientes y varianza Laplaciana $\sigma^2_{\nabla^2 I}$.
5. **Geometría y Llenado**: Relación de aspecto del animal y proporción de área ocupada.
6. **Forma**: Vector Dalal-Triggs HOG normalizado.

In [ ]:
def extract_image_features(image_bgr, target_size=(256, 256)):
    """
    Extrae un vector completo de 349 características discriminativas.
    """
    if image_bgr.shape[:2] != target_size:
        canvas_bgr, bbox = insert_in_black_canvas(image_bgr, target_size=target_size)
    else:
        canvas_bgr = image_bgr
        
    canvas_rgb = cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB)
    canvas_hsv = cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2HSV)
    canvas_gray = cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2GRAY)
    
    # 1. Máscara de primer plano (excluye el fondo negro)
    mask = canvas_gray > 10
    if not np.any(mask):
        mask = np.ones_like(canvas_gray, dtype=bool)
        
    rgb_fg = canvas_rgb[mask]
    hsv_fg = canvas_hsv[mask]
    
    r_m, g_m, b_m = np.mean(rgb_fg, axis=0)
    r_s, g_s, b_s = np.std(rgb_fg, axis=0)
    h_m, s_m, v_m = np.mean(hsv_fg, axis=0)
    h_s, s_s, v_s = np.std(hsv_fg, axis=0)
    
    # Histogramas normalizados
    h_hist = cv2.calcHist([canvas_hsv], [0], mask.astype(np.uint8), [12], [0, 180]).flatten()
    s_hist = cv2.calcHist([canvas_hsv], [1], mask.astype(np.uint8), [8], [0, 256]).flatten()
    v_hist = cv2.calcHist([canvas_hsv], [2], mask.astype(np.uint8), [8], [0, 256]).flatten()
    h_hist /= (np.sum(h_hist) + 1e-6)
    s_hist /= (np.sum(s_hist) + 1e-6)
    v_hist /= (np.sum(v_hist) + 1e-6)
    
    # 2. Pirámide de cuadrantes 2x2
    quad_features = {}
    h_half, w_half = target_size[0] // 2, target_size[1] // 2
    quad_names = ["top_left", "top_right", "bottom_left", "bottom_right"]
    coords = [(0, 0), (0, 1), (1, 0), (1, 1)]
    for name, (r, c) in zip(quad_names, coords):
        q_rgb = canvas_rgb[r*h_half:(r+1)*h_half, c*w_half:(c+1)*w_half]
        q_mask = mask[r*h_half:(r+1)*h_half, c*w_half:(c+1)*w_half]
        if np.any(q_mask):
            q_mean = np.mean(q_rgb[q_mask], axis=0)
            q_std = np.std(q_rgb[q_mask], axis=0)
        else:
            q_mean = [0.0, 0.0, 0.0]
            q_std = [0.0, 0.0, 0.0]
        quad_features[f"{name}_r_mean"] = float(q_mean[0])
        quad_features[f"{name}_g_mean"] = float(q_mean[1])
        quad_features[f"{name}_b_mean"] = float(q_mean[2])
        quad_features[f"{name}_std"] = float(np.mean(q_std))
        
    # 3. Gradientes y Textura
    sobel_x = cv2.Sobel(canvas_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(canvas_gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    grad_mean = float(np.mean(grad_mag[mask]))
    grad_std = float(np.std(grad_mag[mask]))
    laplacian_var = float(cv2.Laplacian(canvas_gray, cv2.CV_64F).var())
    
    # 4. Aspect Ratio y Ocupación
    y_idx, x_idx = np.where(mask)
    if len(y_idx) > 0:
        box_h = np.max(y_idx) - np.min(y_idx) + 1
        box_w = np.max(x_idx) - np.min(x_idx) + 1
        aspect_ratio = float(box_w / (box_h + 1e-6))
    else:
        aspect_ratio = 1.0
    fill_ratio = float(np.sum(mask) / (target_size[0] * target_size[1]))
    
    # 5. HOG Shape Descriptor
    gray_128 = cv2.resize(canvas_gray, (128, 128), interpolation=cv2.INTER_AREA)
    hog_feats = compute_hog_features(gray_128, cell_size=32, n_bins=8)
    
    feature_dict = {
        "r_mean": float(r_m), "g_mean": float(g_m), "b_mean": float(b_m),
        "r_std": float(r_s), "g_std": float(g_s), "b_std": float(b_s),
        "h_mean": float(h_m), "s_mean": float(s_m), "v_mean": float(v_m),
        "h_std": float(h_s), "s_std": float(s_s), "v_std": float(v_s),
        "grad_mean": grad_mean, "grad_std": grad_std,
        "laplacian_var": laplacian_var,
        "aspect_ratio": aspect_ratio, "fill_ratio": fill_ratio
    }
    
    for i in range(12):
        feature_dict[f"h_hist_{i}"] = float(h_hist[i])
    for i in range(8):
        feature_dict[f"s_hist_{i}"] = float(s_hist[i])
        feature_dict[f"v_hist_{i}"] = float(v_hist[i])
        
    feature_dict.update(quad_features)
    
    for i, h_val in enumerate(hog_feats):
        feature_dict[f"hog_{i}"] = float(h_val)
        
    img_64 = cv2.resize(canvas_gray, (64, 64), interpolation=cv2.INTER_AREA)
    img_flat = (img_64.astype(np.float32) / 255.0).flatten()
    
    return feature_dict, canvas_rgb, img_flat

## 5. Extracción Masiva del Dataset y Guardado Tabular

Procesamos los 360 ejemplos del dataset (180 gatos y 180 perros) para consolidar la matriz $\mathbf{X} \in \mathbb{R}^{360 \times 349}$ y el vector de etiquetas $\mathbf{y}$.

In [ ]:
rows = []
print("Extrayendo características para imágenes de gatos...")
for fname in sorted(os.listdir(GATOS_DIR)):
    if fname.endswith(".jpg"):
        img = cv2.imread(os.path.join(GATOS_DIR, fname))
        feats, _, _ = extract_image_features(img)
        feats["label"] = "gatos"
        feats["filename"] = fname
        rows.append(feats)

print("Extrayendo características para imágenes de perros...")
for fname in sorted(os.listdir(PERROS_DIR)):
    if fname.endswith(".jpg"):
        img = cv2.imread(os.path.join(PERROS_DIR, fname))
        feats, _, _ = extract_image_features(img)
        feats["label"] = "perros"
        feats["filename"] = fname
        rows.append(feats)

df_features = pd.DataFrame(rows)
print(f"Dimensiones de la matriz de características extraída: {df_features.shape}")

# Exportar a CSV para disponibilidad en models/
csv_out = os.path.join(DATA_DIR, "features_dataset.csv")
df_features.to_csv(csv_out, index=False)
print(f"Matriz guardada en: {csv_out}")
try:
    display(df_features.head())
except NameError:
    print(df_features.head())